In [3]:
LAST_NAME = "Rosales"
STUDENT_ID = "TUPM-26-3624"
SEED_DIGIT = 4
ID_SUM = 23
NAME_LENGTH = len(LAST_NAME)

TARGET_YEAR = 2015 + SEED_DIGIT

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Target Year:", TARGET_YEAR)

Student: Rosales
Student ID: TUPM-26-3624
Target Year: 2019


In [4]:
import csv
import json
import shutil
from pathlib import Path

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

print("Dataset found:", INPUT_FILE.exists())
print("Output folder ready:", OUTPUT_DIR.exists())
print("Backup folder ready:", BACKUP_DIR.exists())

Dataset found: True
Output folder ready: True
Backup folder ready: True


In [5]:
with open(INPUT_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    print("Column Names:")
    print(reader.fieldnames)

    print("\nFirst 5 Records:")
    for i, row in enumerate(reader):
        if i == 5:
            break
        print(row)

Column Names:
['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']

First 5 Records:
{'Country': 'China', 'Year': '2019', 'Attack Type': 'Phishing', 'Target Industry': 'Education', 'Financial Loss (in Million $)': '80.53', 'Number of Affected Users': '773169', 'Attack Source': 'Hacker Group', 'Security Vulnerability Type': 'Unpatched Software', 'Defense Mechanism Used': 'VPN', 'Incident Resolution Time (in Hours)': '63'}
{'Country': 'China', 'Year': '2019', 'Attack Type': 'Ransomware', 'Target Industry': 'Retail', 'Financial Loss (in Million $)': '62.19', 'Number of Affected Users': '295961', 'Attack Source': 'Hacker Group', 'Security Vulnerability Type': 'Unpatched Software', 'Defense Mechanism Used': 'Firewall', 'Incident Resolution Time (in Hours)': '71'}
{'Country': 'India', 'Year': '2017', 'Attack Type': 'Man

In [6]:
selected_incidents = []
attack_counts = {}
valid_loss_records = []
invalid_loss_count = 0
total_records = 0

with open(INPUT_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1

        if int(row["Year"]) == TARGET_YEAR:
            selected_incidents.append(row)

            attack_type = row["Attack Type"]
            attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1

            try:
                financial_loss = float(row["Financial Loss (in Million $)"])
                valid_loss_records.append(
                    (len(selected_incidents) - 1, row, financial_loss)
                )
            except (ValueError, TypeError):
                invalid_loss_count += 1

print("Total Records:", total_records)
print("Incidents in Target Year:", len(selected_incidents))
print("Incidents by Attack Type:")
for attack_type, count in sorted(attack_counts.items()):
    print(f"{attack_type}: {count}")

print("Invalid Financial Loss Values:", invalid_loss_count)

Total Records: 3000
Incidents in Target Year: 263
Incidents by Attack Type:
DDoS: 48
Malware: 43
Man-in-the-Middle: 44
Phishing: 45
Ransomware: 38
SQL Injection: 45
Invalid Financial Loss Values: 0


In [7]:
total_loss = sum(item[2] for item in valid_loss_records)

if valid_loss_records:
    average_loss = total_loss / len(valid_loss_records)
else:
    average_loss = 0

# Stable sorting keeps original dataset order when losses are tied.
ranked_incidents = sorted(
    valid_loss_records,
    key=lambda item: item[2],
    reverse=True
)

top_five_incidents = ranked_incidents[:5]

print(f"Total Financial Loss: ${total_loss:.2f} million")
print(f"Average Financial Loss: ${average_loss:.2f} million")
print("\nTop 5 Incidents by Financial Loss:")

for rank, (index, incident, loss) in enumerate(top_five_incidents, start=1):
    print(
        f"{rank}. {incident['Country']} | "
        f"{incident['Attack Type']} | "
        f"{incident['Target Industry']} | "
        f"${loss:.2f} million"
    )

Total Financial Loss: $13134.69 million
Average Financial Loss: $49.94 million

Top 5 Incidents by Financial Loss:
1. Germany | Phishing | Healthcare | $99.97 million
2. Russia | Man-in-the-Middle | Retail | $99.53 million
3. UK | Phishing | Education | $99.45 million
4. Germany | Ransomware | Government | $99.30 million
5. Australia | SQL Injection | Government | $99.20 million


In [8]:
FILTERED_CSV = OUTPUT_DIR / f"{LAST_NAME}_cybersecurity_filtered.csv"

fieldnames = [
    "Country",
    "Year",
    "Attack Type",
    "Target Industry",
    "Financial Loss (in Million $)",
    "Number of Affected Users",
    "Attack Source",
    "Security Vulnerability Type",
    "Defense Mechanism Used",
    "Incident Resolution Time (in Hours)"
]

with open(FILTERED_CSV, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_incidents)

print("Filtered CSV created:", FILTERED_CSV)
print("Incidents exported:", len(selected_incidents))

Filtered CSV created: output\Rosales_cybersecurity_filtered.csv
Incidents exported: 263


In [9]:
JSON_FILE = OUTPUT_DIR / f"{LAST_NAME}_cybersecurity_summary.json"

top_five_records = []

for rank, (index, incident, loss) in enumerate(top_five_incidents, start=1):
    record = incident.copy()
    record["Financial Loss (in Million $)"] = loss
    record["Rank"] = rank
    top_five_records.append(record)

summary = {
    "Student": LAST_NAME,
    "Student ID": STUDENT_ID,
    "Target Year": TARGET_YEAR,
    "Total Records in Dataset": total_records,
    "Incidents in Target Year": len(selected_incidents),
    "Incidents by Attack Type": attack_counts,
    "Invalid Financial Loss Values": invalid_loss_count,
    "Total Financial Loss (in Million $)": round(total_loss, 2),
    "Average Financial Loss (in Million $)": round(average_loss, 2),
    "Five Costliest Incidents": top_five_records
}

with open(JSON_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", JSON_FILE)
print("Top incidents included:", len(top_five_records))

JSON summary created: output\Rosales_cybersecurity_summary.json
Top incidents included: 5


In [10]:
BACKUP_FILE = BACKUP_DIR / JSON_FILE.name

shutil.copy2(JSON_FILE, BACKUP_FILE)

backup_verified = JSON_FILE.read_bytes() == BACKUP_FILE.read_bytes()

print("Backup created:", BACKUP_FILE)
print("Backup verified:", backup_verified)

Backup created: backup\Rosales_cybersecurity_summary.json
Backup verified: True


In [ ]:
REPORT_FILE = OUTPUT_DIR / f"{LAST_NAME}_cybersecurity_report.txt"

report_lines = [
    "CYBERSECURITY INCIDENT REPORT",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Target Year: {TARGET_YEAR}",
    "",
    f"Total Records in Dataset: {total_records}",
    f"Incidents in Target Year: {len(selected_incidents)}",
    "",
    "Incidents by Attack Type:"
]

for attack_type, count in sorted(attack_counts.items()):
    report_lines.append(f"  {attack_type}: {count}")

report_lines.extend([
    "",
    f"Invalid Financial Loss Values: {invalid_loss_count}",
    f"Total Financial Loss: ${total_loss:.2f} million",
    f"Average Financial Loss: ${average_loss:.2f} million",
    "",
    "TOP 5 INCIDENTS BY FINANCIAL LOSS"
])

for rank, (index, incident, loss) in enumerate(top_five_incidents, start=1):
    report_lines.append(
        f"{rank}. {incident['Country']} | "
        f"{incident['Attack Type']} | "
        f"{incident['Target Industry']} | "
        f"${loss:.2f} million"
    )

report_lines.extend([
    "",
    "OUTPUT FILES",
    f"Filtered CSV: {FILTERED_CSV}",
    f"JSON Summary: {JSON_FILE}",
    f"JSON Backup: {BACKUP_FILE}",
    f"Backup Verified: {backup_verified}",
    f"Text Report: {REPORT_FILE}"
])

REPORT_FILE.write_text("\n".join(report_lines), encoding="utf-8")

print("Text report created:", REPORT_FILE)
print("Report lines:", len(report_lines))